In [1]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-kzekie7n
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-kzekie7n
  Resolved https://github.com/huggingface/transformers to commit a29df2d916e3b820aecd19d3b5a877abc523ba3c
doneuild dependencies ... 
donetting requirements to build wheel ... 
doneeparing metadata (pyproject.toml) ... 


In [11]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# default: Load the model on the available device(s)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", dtype="auto", device_map="auto"
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen3VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen3-VL-8B-Instruct",
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

IMAGE_PATH = "ocr_beispiel_ausarbeitung.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": (
                "Transcribe this handwritten image line by line in the original order. "
                "Write each line on a new line. "
                "For mathematical expressions, use LaTeX notation (e.g. $x^2 + y^2$). "
                "Only output the transcribed text, nothing else."
            )},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

['Arithmetische Folge\nEine arithmetische Folge ist eine Zahlenreihe, bei der der Unterschied zwischen aufeinanderfolgenden Gliedern konstant ist. Dieser Unterschied wird als $d$ bezeichnet. Die $n$-te Zahl der Folge $a_n$ lässt sich berechnen mit:\n$a_n = a_1 + (n-1) \\cdot d$\nBeispiel:\nWenn die Folge bei 3 beginnt ($a_n=3$) und der Unterschied $d=2$ beträgt, dann ist das 5. G']
